# TP 1 - Ingénierie de consignes modèle


---
## 0. Configuration partagée


In [1]:
import json

from shared.config import ROOT_DIR
from shared.llm_utils import LLMRequest, run_llm
from shared.misc_utils import (
    find_and_parse_json_from_text,
    write_json_file,
)

LOG_DIR = ROOT_DIR / "TP1_travel_planner_LLM" / "logs"

user_query = """
Je veux partir 4 jours à Rome en avril, je n'ai pas encore les dates exactes.
Propose-moi un itinéraire de voyage. Mon budget est de 200 euros pour les sorties et les restaurants.
Je veux éviter les zones trop touristiques et découvrir des lieux plus confidentiels.
"""

#### Fonctions utilitaires pour estimer le coût et afficher l'usage des tokens

In [2]:
def token_estimate_cost_usd(
    token_usage: dict[str, int],
    input_price_per_1m_tokens_usd: float = 0.30,
    output_price_per_1m_tokens_usd: float = 2.50,
) -> float:
    input_cost = token_usage["input_tokens"] * input_price_per_1m_tokens_usd / 1_000_000
    output_cost = token_usage["output_tokens"] * output_price_per_1m_tokens_usd / 1_000_000
    return input_cost + output_cost


def token_print_report(token_usage: dict[str, int]) -> None:
    estimated_cost = token_estimate_cost_usd(token_usage)
    print(
        f"tokens : entrée={token_usage['input_tokens']} | "
        f"sortie={token_usage['output_tokens']} | "
        f"total={token_usage['total_tokens']}"
    )
    print(f"coût estimé (USD) : {estimated_cost:.6f}")


---
### Au préalable

On prépare une base technique pour la logique d'appel LLM

- `project_settings` : configuration partagée du modèle (température, top_p, top_k, max_tokens, budget de réflexion)
- `genai_client` : client Google GenAI authentifié, créé une seule fois
- `LLMRequest` **(TODO)** : classe qui représente les données d'entrée d'un appel LLM (`user_prompt` obligatoire, `system_prompt` optionnel)
- `LLMResponse` **(TODO)** : classe qui représente les données de sortie utiles (texte final, tokens, données brutes)
- `run_llm` **(TODO)** : fonction qui lit la configuration, envoie la requête au modèle et retourne un `LLMResponse`

Les fonctions et classes marquées TODO sont à implémenter dans `shared/llm_utils.py`.

---
## 1. Version 1: requête utilisateur seule


**TODO — Version V1**

Fichier à modifier : `TP1_travel_planner_LLM/1_llm_assistant.ipynb`


Bloc de code qui construit la requête minimale avec `LLMRequest`, appelle `run_llm`, puis enregistre la sortie et les tokens dans `logs/llm_output_v1.txt`


In [ ]:
# TODO : construire la requête minimale V1 sans system prompt
request_v1 = LLMRequest(
    system_prompt=...,
    user_prompt=...,
)
run_result_v1 = ...
final_text_v1 = ...

token_usage_v1 = {
    "input_tokens": ...,
    "output_tokens": ...,
    "total_tokens": ...,
}

log_path_v1 = LOG_DIR / "llm_output_v1.txt"
write_json_file(file_path=log_path_v1, data=run_result_v1.raw_response)

print(final_text_v1)

In [ ]:
token_print_report(token_usage_v1)


---
## 2. Version 2: prompt système structuré


**TODO — Version V2**

Fichier à modifier : `TP1_travel_planner_LLM/1_llm_assistant.ipynb`

L'objectif est d'améliorer la qualité de la réponse avec des instructions claires.<br>
Pour cela, il faut définir :
- **Rôle** : ...
- **Contraintes** : ...
- **Structure** : 4 sections — résumé, itinéraire, budget, conseils.
- **Notes additionnelles** : *utilise cette section pour toute précision ou règle spéciale.*

**But :**
- Produire une réponse complète.
- Rester sous 3000 tokens.


In [ ]:
# TODO : rédiger un system prompt contraint et réutilisable
system_prompt_v2 = """
## Instructions
Tu es ...

Contraintes à respecter:
- ...

Contraintes de style:
- ...

Structure de réponse obligatoire:
1) ...

...
"""

request_v2 = ...
run_result_v2 = ...
final_text_v2 = ...

token_usage_v2 = {
    "input_tokens": int(run_result_v2.input_tokens),
    "output_tokens": int(run_result_v2.output_tokens),
    "total_tokens": int(run_result_v2.total_tokens),
}

log_path_v2 = LOG_DIR / "llm_output_v2.txt"
write_json_file(file_path=log_path_v2, data=run_result_v2.raw_response)

print(final_text_v2)

In [ ]:
token_print_report(token_usage_v2)


---
## 3. Version 3: sortie structurée


**TODO — Version V3**

Fichier à modifier : `TP1_travel_planner_LLM/1_llm_assistant.ipynb`

L'objectif ici est d'avoir une sortie structurée, facile à parser et à traiter.
- Pour cela, il faut imposer un schéma strict JSON (ou XML, ou TOON), en plus de la réponse texte.
- Le prompt système final sera : <br>`system_prompt_v3 = system_prompt_v2 + structured_output_instructions`
- Il faut aussi créer une fonction de parsing pour extraire et parser le bloc JSON depuis le texte final.
- Enfin, on itérera sur ce JSON pour mesurer le coût total estimé et le comparer à la contrainte de budget.


In [ ]:
# TODO : imposer la section JSON finale avec schéma strict
system_prompt_v3 = system_prompt_v2 + """

### Format de réponse
...
"""

request_v3 = ...
run_result_v3 = ...
final_text_v3 = ...

token_usage_v3 = {
    "input_tokens": int(run_result_v3.input_tokens),
    "output_tokens": int(run_result_v3.output_tokens),
    "total_tokens": int(run_result_v3.total_tokens),
}

log_path_v3 = LOG_DIR / "llm_output_v3.txt"
write_json_file(file_path=log_path_v3, data=run_result_v3.raw_response)

print(final_text_v3)

In [ ]:
token_print_report(token_usage_v3)


#### Parser la sortie structurée

**TODO — `find_and_parse_json_from_text`**

Fichier à modifier : `shared/misc_utils.py`

`find_and_parse_json_from_text` : fonction qui récupère le dernier bloc JSON valide dans un texte de réponse, puis retourne un dictionnaire Python ou lève une erreur explicite


In [ ]:
# TODO : parser la sortie texte+JSON sans post-traitement manuel
parsed_json_v3 = ...
print(...)

#### Vérification du budget

Finalement, on peut traiter le JSON parsé pour calculer le coût total estimé des activités proposées, et vérifier que cela respecte la contrainte de budget donnée dans la requête utilisateur.

In [ ]:
...

total_cost_eur = ...
average_per_day = ...

print(f"Total estimé : {total_cost_eur:.2f} EUR")
print(f"Moyenne par jour : {average_per_day:.2f} EUR")
